## Imágenes binarias y morfología

Continuación de [`01_introduccion_opencv.ipynb`](01_introduccion_opencv.ipynb): conversión a escala de grises, umbralización (simple, adaptativa y de Otsu), transformaciones geométricas, y operaciones morfológicas sobre imágenes binarias.

### Escala de grises

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Cargamos la imagen en color
img_color = cv2.imread('img/01_lapices.jpg')

# La convertimos a escala de grises (método luminosidad)
img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
axes[0].set_title('Imagen original')
axes[0].axis('off')
axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title('Escala de grises')
axes[1].axis('off')
plt.show()

### Umbralización (thresholding)

Convertir una imagen en escala de grises a una imagen **binaria** (solo dos valores posibles: 0 o 255), separando el "fondo" del "objeto" según un valor de corte.

#### Umbralización simple

Un único valor de corte fijo para toda la imagen, con distintas variantes de cómo tratar los píxeles por encima/por debajo del umbral.

In [ ]:
img = cv2.imread('img/01_lapices.jpg', cv2.IMREAD_GRAYSCALE)

ret, thresh1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
ret, thresh2 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
ret, thresh3 = cv2.threshold(img, 127, 255, cv2.THRESH_TRUNC)
ret, thresh4 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)
ret, thresh5 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO_INV)

titles = ['GRAY', 'BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV']
images = [img, thresh1, thresh2, thresh3, thresh4, thresh5]

plt.figure(figsize=(10, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1), plt.imshow(images[i], 'gray', vmin=0, vmax=255)
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.show()

#### Umbralización adaptativa

Un solo umbral fijo no siempre funciona bien si la iluminación varía a lo largo de la imagen (por ejemplo, un documento escaneado con sombras). La umbralización **adaptativa** calcula un umbral distinto para cada región.

In [ ]:
img = cv2.imread('img/02_escaneado.png', cv2.IMREAD_GRAYSCALE)
img = cv2.medianBlur(img, 5)

ret, th1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
th2 = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 2)
th3 = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

titles = ['Imagen original', 'Umbral global (v=127)', 'Umbral adaptativo (media)', 'Umbral adaptativo (gaussiano)']
images = [img, th1, th2, th3]

plt.figure(figsize=(10, 8))
for i in range(4):
    plt.subplot(2, 2, i + 1), plt.imshow(images[i], 'gray')
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.show()

#### Umbralización de Otsu

En vez de elegir el umbral "a mano" (el `127` de los ejemplos anteriores), el método de **Otsu** lo calcula automáticamente a partir del histograma de la imagen — muy útil con imágenes bimodales (fondo y objeto bien diferenciados) y ruidosas.

In [ ]:
img = cv2.imread('img/04_noisy.png', cv2.IMREAD_GRAYSCALE)

# Umbralización global común (umbral fijo)
ret1, th1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

# Umbralización de Otsu
ret2, th2 = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Umbralización de Otsu luego de un filtro Gaussiano (reduce el ruido antes de umbralizar)
blur = cv2.GaussianBlur(img, (5, 5), 0)
ret3, th3 = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

images = [img, 0, th1, img, 0, th2, blur, 0, th3]
titles = ['Imagen con ruido', 'Histograma', 'Umbral global (v=127)',
          'Imagen con ruido', 'Histograma', 'Umbral de Otsu',
          'Imagen filtrada (Gaussiano)', 'Histograma', 'Umbral de Otsu']

plt.figure(figsize=(10, 10))
for i in range(3):
    plt.subplot(3, 3, i * 3 + 1), plt.imshow(images[i * 3], 'gray')
    plt.title(titles[i * 3]), plt.xticks([]), plt.yticks([])
    plt.subplot(3, 3, i * 3 + 2), plt.hist(images[i * 3].ravel(), 256)
    plt.title(titles[i * 3 + 1]), plt.xticks([]), plt.yticks([])
    plt.subplot(3, 3, i * 3 + 3), plt.imshow(images[i * 3 + 2], 'gray')
    plt.title(titles[i * 3 + 2]), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Transformaciones geométricas

Operaciones que cambian la posición, orientación o forma de una imagen: traslación, rotación, escalado y corrección de perspectiva. Todas se implementan aplicando una **matriz de transformación** sobre las coordenadas de cada píxel.

#### Traslación

In [ ]:
import numpy as np

img = cv2.imread('img/05_messi.jpg')

# Matriz de traslación: mueve la imagen 200 píxeles a la derecha y 100 hacia abajo
M = np.float32([[1, 0, 200], [0, 1, 100]])

height, width = img.shape[:2]
translated_img = cv2.warpAffine(img, M, (width, height))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(translated_img, cv2.COLOR_BGR2RGB)); axes[1].set_title('Trasladada'); axes[1].axis('off')
plt.show()

#### Rotación

In [ ]:
img = cv2.imread('img/05_messi.jpg')
height, width = img.shape[:2]

angle = 30            # grados, sentido antihorario
center = (width // 2, height // 2)

M = cv2.getRotationMatrix2D(center, angle, 1.0)
rotated_img = cv2.warpAffine(img, M, (width, height))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(rotated_img, cv2.COLOR_BGR2RGB)); axes[1].set_title('Rotada 30°'); axes[1].axis('off')
plt.show()

#### Escalado

Al agrandar o achicar una imagen, OpenCV necesita "inventar" valores para los píxeles nuevos (o descartar información al achicar). Cómo lo hace depende del método de **interpolación** elegido.

In [ ]:
img = cv2.imread('img/05_messi.jpg')
height, width = img.shape[:2]
nuevo_ancho, nueva_altura = width * 2, height * 2

interpolations = [cv2.INTER_NEAREST, cv2.INTER_LINEAR, cv2.INTER_CUBIC, cv2.INTER_AREA, cv2.INTER_LANCZOS4]
interp_names = ['INTER_NEAREST', 'INTER_LINEAR', 'INTER_CUBIC', 'INTER_AREA', 'INTER_LANCZOS4']

plt.figure(figsize=(16, 8))
for i, interp in enumerate(interpolations):
    resized_img = cv2.resize(img, (nuevo_ancho, nueva_altura), interpolation=interp)
    plt.subplot(2, 3, i + 1)
    plt.imshow(cv2.cvtColor(resized_img, cv2.COLOR_BGR2RGB))
    plt.title(interp_names[i])
    plt.axis('off')
plt.suptitle('Mismo escalado 2x con distintos métodos de interpolación', fontsize=14)
plt.tight_layout()
plt.show()

#### Corrección de perspectiva

Dados 4 puntos de una imagen (por ejemplo, las 4 esquinas de una fachada fotografiada en ángulo) y los 4 puntos donde queremos que terminen, `cv2.getPerspectiveTransform` calcula la matriz que "endereza" esa región.

> 💡 En la versión original de este ejercicio, los 4 puntos se elegían haciendo clic con el mouse sobre la imagen (`cv2.setMouseCallback`). Eso abre una ventana nativa del sistema operativo y espera la interacción del usuario, algo que no es posible dentro de una notebook — así que acá los puntos quedan **fijos**, elegidos de antemano sobre las esquinas de la fachada central.

In [ ]:
img = cv2.imread('img/06_notredame.jpg')
height, width = img.shape[:2]
print(f'Tamaño de la imagen: {width}x{height}')

# Esquinas de la fachada central, elegidas a mano (sup. izq., sup. der., inf. der., inf. izq.)
points = np.array([[345, 170], [855, 170], [885, 600], [315, 600]], dtype=np.float32)

# Dibujamos los puntos elegidos sobre la imagen, para verificar que caen donde esperamos
img_puntos = img.copy()
for x, y in points:
    cv2.circle(img_puntos, (int(x), int(y)), 8, (0, 255, 0), -1)
plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(img_puntos, cv2.COLOR_BGR2RGB))
plt.title('Puntos de origen elegidos')
plt.axis('off')
plt.show()

In [ ]:
# Puntos de destino: un rectángulo (así es como se vería la fachada "de frente")
dest_width, dest_height = 300, 400
dest_points = np.array([[0, 0], [dest_width - 1, 0], [dest_width - 1, dest_height - 1], [0, dest_height - 1]], dtype=np.float32)

matrix = cv2.getPerspectiveTransform(points, dest_points)
warped = cv2.warpPerspective(img, matrix, (dest_width, dest_height))

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)); axes[1].set_title('Perspectiva corregida'); axes[1].axis('off')
plt.tight_layout()
plt.show()

### Operaciones morfológicas

Operaciones que modifican la forma de los objetos en una imagen **binaria**, usando un *kernel* (elemento estructurante) que recorre la imagen:

- **Erosión**: "encoge" los objetos blancos (elimina detalles finos y separa objetos pegados).
- **Dilatación**: "agranda" los objetos blancos (rellena huecos pequeños).
- **Apertura** (erosión + dilatación): elimina ruido pequeño sin cambiar mucho el tamaño general de los objetos.
- **Cierre** (dilatación + erosión): rellena huecos pequeños dentro de los objetos.
- **Gradiente morfológico**: diferencia entre dilatación y erosión — resalta el contorno de los objetos.

In [ ]:
img = cv2.imread('img/09_cebra.jpeg', cv2.IMREAD_GRAYSCALE)

_, binary_img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

kernel = np.ones((5, 5), np.uint8)

erosion = cv2.erode(binary_img, kernel, iterations=1)
dilation = cv2.dilate(binary_img, kernel, iterations=1)
opening = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(binary_img, cv2.MORPH_CLOSE, kernel)
gradient = cv2.morphologyEx(binary_img, cv2.MORPH_GRADIENT, kernel)

titles = ['Imagen binaria', 'Erosión', 'Dilatación', 'Apertura', 'Cierre', 'Gradiente']
images = [binary_img, erosion, dilation, opening, closing, gradient]

plt.figure(figsize=(14, 9))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.title(titles[i])
    plt.imshow(images[i], cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()